In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def setup_environment(output_dir: str):
    """Creates the output directory and sets visualization styling."""
    os.makedirs(output_dir, exist_ok=True)
    sns.set_theme(style="whitegrid")
    plt.rcParams["figure.figsize"] = (10, 6)
    plt.rcParams["axes.labelsize"] = 12
    plt.rcParams["axes.titlesize"] = 14


def plot_class_imbalance(df: pd.DataFrame, target_col: str, output_dir: str):
    """Visualizes the target variable distribution."""
    plt.figure(figsize=(6, 5))
    sns.countplot(x=target_col, data=df, palette="viridis")
    plt.title("Distribution of Customer Churn")
    plt.xlabel("Churn Status (0 = Retained, 1 = Churned)")
    plt.ylabel("Number of Customers")
    
    save_path = os.path.join(output_dir, "01_class_imbalance.png")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


def plot_numerical_distributions(df: pd.DataFrame, num_cols: list, target_col: str, output_dir: str):
    """Visualizes how numerical features shift between churned and retained users."""
    core_numerical = [c for c in num_cols if c not in [target_col, "id"]]
    
    for i, col in enumerate(core_numerical, start=2):
        plt.figure(figsize=(10, 5))
        sns.kdeplot(
            data=df,
            x=col,
            hue=target_col,
            fill=True,
            common_norm=False,
            palette="Set1",
            alpha=0.5,
        )
        plt.title(f"Impact of {col.title()} on Customer Churn Risk")
        plt.xlabel(col.title())
        plt.ylabel("Density Profile")
        
        save_path = os.path.join(output_dir, f"0{i}_{col}_distribution.png")
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()


def plot_categorical_churn_rates(df: pd.DataFrame, cat_cols: list, target_col: str, output_dir: str):
    """Visualizes the average churn probability per category."""
    for i, col in enumerate(cat_cols[:3], start=5):
        plt.figure(figsize=(10, 5))
        churn_rates = df.groupby(col)[target_col].mean().reset_index()
        churn_rates = churn_rates.sort_values(by=target_col, ascending=False)

        sns.barplot(
            x=target_col,
            y=col,
            data=churn_rates,
            palette="coolwarm",
            hue=col,
            legend=False,
        )
        plt.title(f"Churn Probability Rate by {col.title()}")
        plt.xlabel("Average Churn Percentage (0.0 - 1.0)")
        plt.ylabel(col.title())
        
        # Add a baseline marker
        plt.axvline(
            df[target_col].mean(),
            color="red",
            linestyle="--",
            label="Global Baseline Churn Rate",
        )
        plt.legend()
        
        save_path = os.path.join(output_dir, f"0{i}_{col}_churn_rates.png")
        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()


def plot_correlation_matrix(df: pd.DataFrame, num_cols: list, output_dir: str):
    """Visualizes the Pearson correlation across all numerical features."""
    plt.figure(figsize=(10, 8))
    corr_matrix = df[num_cols].corr()

    # Mask the upper triangle for a cleaner look
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )

    plt.title("Numerical Features - Correlation Heatmap Matrix", pad=20)
    
    save_path = os.path.join(output_dir, "99_correlation_matrix.png")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()


def run_eda_pipeline(data_path: str, output_dir: str):
    """Executes the complete EDA pipeline."""
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Cleaned dataset not found at: {data_path}")
        
    df = pd.read_csv(data_path)


    setup_environment(output_dir)

    # Dynamically identify feature groups
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    target_col = [c for c in df.columns if "churn" in c.lower()]

    if not target_col:
        print("Could not find a target column containing the word 'churn'. Exiting.")
        return
        
    target_col = target_col[0]
    
    # Execute visualization functions
    plot_class_imbalance(df, target_col, output_dir)
    plot_numerical_distributions(df, num_cols, target_col, output_dir)
    plot_categorical_churn_rates(df, cat_cols, target_col, output_dir)
    plot_correlation_matrix(df, num_cols, output_dir)
    


if __name__ == "__main__":
    # Define file paths
    CLEANED_DATA_PATH = "../data/cleaned_customer_churn.csv"
    OUTPUT_DIRECTORY = "visualizations"

    try:
        run_eda_pipeline(CLEANED_DATA_PATH, OUTPUT_DIRECTORY)
    except Exception as e:
        print(f"Error during EDA execution: {str(e)}")

<string>:129: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
<string>:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

